In [18]:
import sys
sys.path.append('WTM')

import pickle as p
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import new_ds
from new_ds import mm_Dataset
from WTM_model import WTM
from utils import calc_topic_uniqueness
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
pd.set_option('display.max_rows', None)
import pickle

import seaborn as sns
import time

import argparse
import multiprocessing as mp

def my_softmax(df):
    
    maxes = df.max(axis=1)
    diffs = df.sub(maxes, axis=0)
    num = np.exp(diffs)
    denom = num.sum(axis=1)
    out = num.div(denom, axis=0)
    return out

def load_model(model_prefix, docSet):
    checkpoint = torch.load(f'WTM_checkpoints/{model_prefix}')
    param = checkpoint["param"]
    model = WTM(**param)
    model.load_model(checkpoint["net"])
    model.id2token = {v: k for k,v in docSet.dictionary.token2id.items()}
    return model

def get_embeds(model, docSet, labels_fname = None):
    embeds = model.get_embed(train_data=docSet, num=1000)

    if labels_fname:
        with open(labels_fname, 'r') as file:
            labels_list = [line.strip() for line in file.readlines()]
        out_df = pd.DataFrame(embeds, index=labels_list)
        return out_df
        
    return pd.DataFrame(embeds)

def calculate_phis(docSet, model):
    out_dict = {}
    all_phi_raw = model.get_topic_word_dist(normalize=False)
    headers = np.concatenate([expmat.columns for expmat in docSet.expmats.values()])
    all_phi_raw = pd.DataFrame(all_phi_raw, columns = headers)
    for data_type in np.unique(docSet.label_type):
        type_raw = all_phi_raw.iloc[:, docSet.label_type==data_type]
        out_dict[data_type] = my_softmax(type_raw)
    return out_dict

def get_samp_raw_linkage(model, docSet, embeds, phis, name1, name2, samp):

    n1 = name1 if type(name1)==int else model.token2d(name1)
    n2 =  name2 if type(name2)==int else model.token2d(name2)

    n1_type = docSet.label_type[n1]
    n2_type = docSet.label_type[n2]

    if (docSet.expmats[n1_type].iloc[samp,:].loc[n1] == 0) or (docSet.expmats[n2_type].iloc[samp,:].loc[n2] == 0):
        return 0

    num = 0
    for t in range(model.n_topic):
        topic_prop = embeds.iloc[samp, t]
        n1_phi = phis[n1_type].loc[t, n1]
        n2_phi = phis[n2_type].loc[t, n2]
        num += topic_prop*n1_phi*n2_phi

    denom = np.linalg.norm(phis[n1_type].loc[:, n1]) * np.linalg.norm(phis[n1_type].loc[:, n1])

    return num/denom

# get_samp_raw_linkage(curr_model, covid_docSet, covid_embeds, 0, 2, 0)
# get_raw_linkage(curr_model, covid_docSet, 0, 2)

def linkage_calc_mp_wrapper(params):
    start = time.time()
    s, i_range, j_range, curr_model, curr_docSet, curr_embeds, curr_phis = params

    results = []
    for i in range(i_range):
        for j in range(j_range):
            link_score = get_samp_raw_linkage(curr_model, curr_docSet, curr_embeds, curr_phis, i, j, s)
            results.append([s, i, j, link_score])
    print(f'Finished with Samp {s}, took {round(time.time() - start)}')    
    return results

In [19]:
def new_calc_link_sample(params):
    s, type1, type2, curr_model, curr_docSet, curr_embeds, curr_phis = params

    I_x = curr_docSet.expbool[type1].iloc[s, :].astype('float32').values
    I_y = curr_docSet.expbool[type2].iloc[s, :].astype('float32').values
    theta = curr_embeds.iloc[s, :]
    phi_x = curr_phis[type1]
    phi_y = curr_phis[type2]

    embed_weighted = phi_x.mul(theta, axis=0)
    # print(embed_weighted.shape)
    
    phis_mult = embed_weighted.transpose().dot(phi_y)
    # print(phis_mult.shape)

    x_masked = phis_mult.mul(I_x, axis=0)
    xy_masked = x_masked.mul(I_y, axis=1)
    
    phi_x_norms = phi_x.apply(np.linalg.norm, axis=0) + 1e-10# apply norm on each column
    # print(np.sort(np.unique(phi_x_norms)))
    # print(phi_x_norms.shape)
    
    phi_y_norms = phi_y.apply(np.linalg.norm, axis=0) + 1e-10 # apply norm down each column
    # print(np.sort(np.unique(phi_y_norms)))
    # print(phi_y_norms.shape)

    normed = xy_masked.div(phi_x_norms, axis=0).div(phi_y_norms, axis=1)
    
    return normed

In [20]:
def calc_link_all(params):
    type1, type2, curr_model, curr_docSet, curr_embeds, curr_phis = params

    phi_x = curr_phis[type1]
    phi_y = curr_phis[type2]
    phi_x_norms = phi_x.apply(np.linalg.norm, axis=0) + 1e-10
    phi_y_norms = phi_y.apply(np.linalg.norm, axis=0) + 1e-10

    scores = phi_x.transpose().dot(phi_y).div(phi_x_norms, axis=0).div(phi_y_norms, axis=1)
    
    return scores
    

In [47]:
# model_path = 'ibs_model_tp10_a0.1_lr0.001/100.ckpt' #'covid_tp10_a0.05_lr0.0005/500.ckpt'
# docSet_path = 'ibs_data/ibs_docDataset.pkl'

# model_path, docSet_path = 'covid_tp10_a0.05_lr0.0005/500.ckpt', 'covid_data/covid_docDataset.pkl' # covid
# model_path, docSet_path = 'crc_model_tp10_a0.1_lr0.001/100.ckpt', 'crc_data/crc_docDataset.pkl'
# model_path, docSet_path = 'ibd_model_tp10_a0.1_lr0.001/100.ckpt', 'ibd_data/ibd_docDataset.pkl'
model_path, docSet_path = 'ibs_model_tp10_a0.1_lr0.001/100.ckpt', 'ibs_data/ibs_docDataset.pkl'

with open(docSet_path, 'rb') as handle:
    curr_docSet = pickle.load(handle)

curr_model = load_model(model_path, curr_docSet)

curr_embeds = get_embeds(curr_model, curr_docSet)

curr_phis = calculate_phis(curr_docSet, curr_model)

In [48]:
# for per-sample linkages
# linkage_lists = []
# for i in range(curr_docSet.numDocs):
#     linkage_lists.append(new_calc_link_sample([i, 'gene', 'microbeR', curr_model, curr_docSet, curr_embeds, curr_phis]))

# linkage_df = pd.concat(linkage_lists, keys = list(range(curr_docSet.numDocs)))
# linkage_df.to_pickle('ibs_data/raw_linkages_df.p')

In [49]:
# for overall linkages
linkages = calc_link_all(['gene', 'microbeR', curr_model, curr_docSet, curr_embeds, curr_phis])
linkages.shape

(17841, 910)

In [50]:
linkages.to_pickle('ibs_data/raw_linkages_overall.p')